# 05 Candidate Model Training and Tuning

## 1. Objective and Scope

This notebook identifies the best configuration within each of three candidate regression model families for AFL matchday attendance forecasting:  
  
- a regularised linear model  
- a random forest regressor  
- a gradient boosting regressor  
  
Only the historical match data assigned to the `training` partition are used for model development. Hyperparameters are evaluated through time-aware cross-validation, ensuring that each validation fold occurs after its corresponding training data. Data preprocessing is incorporated into each model pipeline to prevent information leakage during cross-validation.

The purpose of this notebook is not to select the final forecasting model. Instead, it produces one tuned configuration for each model family and a comparable cross-validation summary for use in Notebook 06. The 2024 validation partition, the 2025 locked test partition, and the 2026 scoring dataset remain untouched.

## 2. Load and Prepare the Training Data

### 2.1 Import Libraries and Set Configuration

This section imports the libraries required for data preparation, preprocessing, model training, hyperparameter tuning, and output management. A fixed random seed is used to support reproducible model results.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Reproducibility and parallel processing
RANDOM_STATE = 42
N_JOBS = -1


# Display configuration
pd.set_option("display.max_columns", None)

### 2.2 Load the Historical Model Dataset

The model-ready historical dataset created in Notebook 04 is loaded from `data/processed/`. The `match_date` field is parsed as a datetime value, and a compact summary is produced to confirm that the expected dataset has been loaded successfully.

The separate 2026 scoring dataset is not loaded because future-fixture prediction is outside the scope of this notebook.

In [2]:
# Resolve the project root whether the notebook starts from the
# repository root or from the notebooks directory.
project_root = Path.cwd().resolve()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent


# Define the historical model dataset path.
historical_data_path = (
    project_root
    / "data"
    / "processed"
    / "historical_model_dataset.csv"
)

if not historical_data_path.exists():
    raise FileNotFoundError(
        f"Historical model dataset was not found: {historical_data_path}"
    )


# Load the model-ready historical dataset.
historical_df = pd.read_csv(
    historical_data_path,
    parse_dates=["match_date"],
)


# Summarise the loaded dataset.
load_summary = pd.DataFrame(
    [
        {
            "file_name": historical_data_path.name,
            "rows": len(historical_df),
            "columns": historical_df.shape[1],
            "earliest_match": historical_df["match_date"].min().date(),
            "latest_match": historical_df["match_date"].max().date(),
        }
    ]
)

load_summary

,file_name,rows,columns,earliest_match,latest_match
0,historical_model_dataset.csv,2297,23,2013-03-22,2025-09-27


### 2.3 Isolate the Training Partition

Notebook 04 assigned each historical match to a fixed project-level data role. This section isolates the 1,865 matches in the training partition, covering the 2013–2019 and 2022–2023 seasons.

No new project-level split is created here. The season-based expanding-window folds used for internal hyperparameter tuning are defined later in Section 3.2. The 2024 validation partition and the 2025 locked test partition remain untouched.

In [3]:
# Isolate and chronologically order the fixed training partition.
training_df = (
    historical_df
    .loc[historical_df["dataset_role"].eq("training")]
    .copy()
    .sort_values("match_date", kind="stable")
    .reset_index(drop=True)
)


# Confirm the expected training coverage.
training_seasons = sorted(
    training_df["match_date"].dt.year.unique().tolist()
)

expected_training_seasons = [
    2013, 2014, 2015, 2016, 2017, 2018, 2019, 2022, 2023
]

if len(training_df) != 1865:
    raise ValueError(
        f"Expected 1,865 training matches, but found {len(training_df):,}."
    )

if training_seasons != expected_training_seasons:
    raise ValueError(
        f"Unexpected training seasons: {training_seasons}"
    )


training_partition_summary = pd.DataFrame(
    [
        {
            "rows": len(training_df),
            "seasons": ", ".join(map(str, training_seasons)),
            "earliest_match": training_df["match_date"].min().date(),
            "latest_match": training_df["match_date"].max().date(),
        }
    ]
)

training_partition_summary

,rows,seasons,earliest_match,latest_match
0,1865,"2013, 2014, 2015, 2016, 2017, 2018, 2019, 2022...",2013-03-22,2023-09-30


### 2.4 Define Features and Target

Before constructing the modelling inputs, the schema of the training partition is reviewed to distinguish predictive features from the attendance target and supporting metadata.

The summary below reports the data type, missing-value count, and number of unique values for each field. This check supports the explicit definition of numerical and categorical predictors while ensuring that identifiers, dates, dataset roles, and other non-predictive fields are excluded from the model inputs.

In [4]:
schema_summary = pd.DataFrame(
    {
        "column_name": training_df.columns,
        "dtype": training_df.dtypes.astype(str).to_numpy(),
        "missing_values": training_df.isna().sum().to_numpy(),
        "unique_values": training_df.nunique(dropna=True).to_numpy(),
    }
)

schema_summary

,column_name,dtype,missing_values,unique_values
0,source_game_id,str,0,1865
1,match_id,str,0,1865
2,match_date,datetime64[us],0,795
3,start_time,str,0,54
4,dataset_role,str,0,1
5,attendance,int64,0,1837
6,season_year,int64,0,9
7,start_hour,float64,0,54
8,home_team_last_5_attendance_mean,float64,9,1851
9,away_team_last_5_attendance_mean,float64,9,1850


The model features are defined explicitly rather than selected automatically from data types. Unique match identifiers, the raw match date and start time, and the dataset-role label are retained for data management but excluded from the feature matrix.

The remaining 17 pre-match fields are divided into nine numerical features and eight categorical features. `match_month` is treated as categorical because month numbers do not represent a simple linear relationship. The two Boolean indicators are also processed as discrete categorical states.

In [5]:
# Define the regression target and supporting metadata.
TARGET_COLUMN = "attendance"

METADATA_COLUMNS = [
    "source_game_id",
    "match_id",
    "match_date",
    "start_time",
    "dataset_role",
]


# Define the numerical features.
NUMERICAL_FEATURES = [
    "season_year",
    "start_hour",
    "home_team_last_5_attendance_mean",
    "away_team_last_5_attendance_mean",
    "venue_last_10_attendance_mean",
    "home_team_last_5_win_rate",
    "away_team_last_5_win_rate",
    "home_team_last_5_score_margin_mean",
    "away_team_last_5_score_margin_mean",
]


# Define the categorical features.
CATEGORICAL_FEATURES = [
    "round_label",
    "home_team",
    "away_team",
    "venue_name",
    "match_month",
    "match_day_of_week",
    "is_night_match",
    "is_school_holiday",
]


# Combine the feature groups in a fixed order.
FEATURE_COLUMNS = NUMERICAL_FEATURES + CATEGORICAL_FEATURES


# Validate the feature definition.
if len(FEATURE_COLUMNS) != 17:
    raise ValueError(
        f"Expected 17 features, but found {len(FEATURE_COLUMNS)}."
    )

if len(FEATURE_COLUMNS) != len(set(FEATURE_COLUMNS)):
    raise ValueError("Duplicate feature names were detected.")

metadata_overlap = sorted(
    set(FEATURE_COLUMNS).intersection(METADATA_COLUMNS)
)

if metadata_overlap:
    raise ValueError(
        f"Metadata fields were included as features: {metadata_overlap}"
    )

if TARGET_COLUMN in FEATURE_COLUMNS:
    raise ValueError("The target column was included in the feature matrix.")


# Confirm that all required columns are available.
required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_columns = sorted(
    set(required_columns) - set(training_df.columns)
)

if missing_columns:
    raise KeyError(
        f"Required modelling columns are missing: {missing_columns}"
    )


# Construct the model-development feature matrix and target vector.
X_train = training_df.loc[:, FEATURE_COLUMNS].copy()

y_train = (
    training_df
    .loc[:, TARGET_COLUMN]
    .astype("float64")
    .copy()
)


# Validate the resulting model inputs.
if len(X_train) != len(y_train):
    raise ValueError(
        "The feature matrix and target vector have different row counts."
    )

if y_train.isna().any():
    raise ValueError("The training target contains missing values.")


# Summarise the model inputs.
model_input_summary = pd.DataFrame(
    [
        {
            "training_rows": len(X_train),
            "features": X_train.shape[1],
            "numerical_features": len(NUMERICAL_FEATURES),
            "categorical_features": len(CATEGORICAL_FEATURES),
            "missing_feature_values": int(
                X_train.isna().sum().sum()
            ),
            "missing_target_values": int(y_train.isna().sum()),
            "target": TARGET_COLUMN,
        }
    ]
)

model_input_summary

,training_rows,features,numerical_features,categorical_features,missing_feature_values,missing_target_values,target
0,1865,17,9,8,34,0,attendance


## 3. Define the Model Tuning Framework

### 3.1 Preprocessing Pipeline

All preprocessing steps are placed inside scikit-learn pipelines so that they are fitted separately within the training portion of each time-aware cross-validation fold. This prevents information from a fold's validation season from influencing feature preparation.

Missing numerical values are imputed using the median calculated from the relevant training fold. Numerical features are then standardised for the regularised linear model because its coefficients and penalty are sensitive to feature scale. Scaling is not applied to the tree-based models because their split decisions do not require it.

Categorical features are imputed using the most frequent value and one-hot encoded. Previously unseen categories are ignored so that future matches containing a new team–venue combination can still be processed.

In [6]:
def build_preprocessor(scale_numeric):
    """Create a fresh preprocessing pipeline for a model family."""

    numerical_steps = [
        ("imputer", SimpleImputer(strategy="median")),
    ]

    if scale_numeric:
        numerical_steps.append(
            ("scaler", StandardScaler())
        )

    numerical_pipeline = Pipeline(
        steps=numerical_steps
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "one_hot_encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numerical",
                numerical_pipeline,
                NUMERICAL_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                CATEGORICAL_FEATURES,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


# Create separate preprocessing objects for the model families.
linear_preprocessor = build_preprocessor(
    scale_numeric=True
)

tree_preprocessor = build_preprocessor(
    scale_numeric=False
)


# Summarise the preprocessing design.
preprocessing_summary = pd.DataFrame(
    [
        {
            "model_group": "Regularised linear model",
            "numerical_imputation": "Median",
            "numerical_scaling": "StandardScaler",
            "categorical_imputation": "Most frequent",
            "categorical_encoding": "One-hot",
        },
        {
            "model_group": "Tree-based models",
            "numerical_imputation": "Median",
            "numerical_scaling": "None",
            "categorical_imputation": "Most frequent",
            "categorical_encoding": "One-hot",
        },
    ]
)

preprocessing_summary

,model_group,numerical_imputation,numerical_scaling,categorical_imputation,categorical_encoding
0,Regularised linear model,Median,StandardScaler,Most frequent,One-hot
1,Tree-based models,Median,None,Most frequent,One-hot


### 3.2 Time-Aware Cross-Validation

Random cross-validation is not appropriate because it could allow later seasons to influence predictions for earlier matches. Instead, six season-based expanding-window folds are constructed within the fixed training partition.

The first fold trains on the 2013–2015 seasons and validates on 2016. Each subsequent fold adds the previous validation season to the training window and evaluates the next available season. The sequence moves directly from 2019 to 2022 because the excluded 2020–2021 pandemic seasons are not present in the modelling dataset.

These internal validation seasons remain part of the project-level training partition. The separate 2024 validation and 2025 locked test partitions are not used in this notebook.

In [7]:
# Define the complete seasons used as internal CV validation folds.
CV_VALIDATION_SEASONS = [
    2016,
    2017,
    2018,
    2019,
    2022,
    2023,
]


# Construct reusable positional-index splits for GridSearchCV.
time_aware_cv_splits = []
cv_summary_records = []

for fold_number, validation_season in enumerate(
    CV_VALIDATION_SEASONS,
    start=1,
):
    train_mask = training_df["season_year"].lt(
        validation_season
    )

    validation_mask = training_df["season_year"].eq(
        validation_season
    )

    train_indices = np.flatnonzero(
        train_mask.to_numpy()
    )

    validation_indices = np.flatnonzero(
        validation_mask.to_numpy()
    )

    if len(train_indices) == 0:
        raise ValueError(
            f"Fold {fold_number} has no training observations."
        )

    if len(validation_indices) == 0:
        raise ValueError(
            f"Fold {fold_number} has no validation observations."
        )

    latest_training_date = training_df.iloc[
        train_indices
    ]["match_date"].max()

    earliest_validation_date = training_df.iloc[
        validation_indices
    ]["match_date"].min()

    if latest_training_date >= earliest_validation_date:
        raise ValueError(
            f"Fold {fold_number} violates chronological order."
        )

    training_seasons = sorted(
        training_df.iloc[
            train_indices
        ]["season_year"].unique().tolist()
    )

    time_aware_cv_splits.append(
        (train_indices, validation_indices)
    )

    cv_summary_records.append(
        {
            "fold": fold_number,
            "training_seasons": ", ".join(
                map(str, training_seasons)
            ),
            "validation_season": validation_season,
            "training_rows": len(train_indices),
            "validation_rows": len(validation_indices),
        }
    )


if len(time_aware_cv_splits) != 6:
    raise ValueError(
        f"Expected 6 CV folds, but created "
        f"{len(time_aware_cv_splits)}."
    )


cv_split_summary = pd.DataFrame(
    cv_summary_records
)

cv_split_summary

,fold,training_seasons,validation_season,training_rows,validation_rows
0,1,"2013, 2014, 2015",2016,617,207
1,2,"2013, 2014, 2015, 2016",2017,824,206
2,3,"2013, 2014, 2015, 2016, 2017",2018,1030,206
3,4,"2013, 2014, 2015, 2016, 2017, 2018",2019,1236,206
4,5,"2013, 2014, 2015, 2016, 2017, 2018, 2019",2022,1442,207
5,6,"2013, 2014, 2015, 2016, 2017, 2018, 2019, 2022",2023,1649,216


### 3.3 Tuning Metric

Mean Absolute Error (MAE) is used as the primary tuning metric because it expresses the typical forecasting error directly in attendee numbers. This makes the result straightforward to interpret for matchday planning.

Each hyperparameter configuration is evaluated using its mean MAE across the six time-aware validation folds. The configuration with the lowest mean CV MAE is retained within each model family. The standard deviation of fold-level MAE is also recorded to assess performance stability across seasons.

Root Mean Squared Error (RMSE) is retained as a secondary diagnostic because it places greater weight on large forecasting errors. It does not determine the winning configuration in this notebook.

Scikit-learn represents error-based scorers as negative values because `GridSearchCV` maximises scores. These values will be converted back to positive MAE and RMSE when the results are reported.

In [8]:
# Define the metrics calculated during hyperparameter tuning.
TUNING_SCORING = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
}


# Select the metric used to rank configurations and refit each search.
REFIT_METRIC = "mae"


# Summarise the role of each tuning metric.
tuning_metric_summary = pd.DataFrame(
    [
        {
            "metric": "MAE",
            "grid_search_scorer": TUNING_SCORING["mae"],
            "role": "Primary tuning metric",
            "selection_rule": "Lowest mean CV MAE",
        },
        {
            "metric": "RMSE",
            "grid_search_scorer": TUNING_SCORING["rmse"],
            "role": "Secondary diagnostic",
            "selection_rule": "Not used to select configuration",
        },
    ]
)

tuning_metric_summary

,metric,grid_search_scorer,role,selection_rule
0,MAE,neg_mean_absolute_error,Primary tuning metric,Lowest mean CV MAE
1,RMSE,neg_root_mean_squared_error,Secondary diagnostic,Not used to select configuration


## 4. Configure Candidate Model Families

### 4.1 Candidate Model Selection Rationale

Three candidate model families are configured to represent different relationships between the pre-match features and attendance.

- Ridge Regression provides a regularised linear benchmark.  
- Random Forest can capture nonlinear relationships and interactions through an ensemble of independently fitted decision trees.  
- Histogram Gradient Boosting builds trees sequentially so that later trees focus on correcting earlier prediction errors.

Each model is paired with the appropriate preprocessing pipeline defined in Section 3.1. The search grids are intentionally compact so that meaningful alternatives can be compared across all six time-aware folds without making the tuning process unnecessarily expensive.

Model families are not compared using the 2024 validation partition in this notebook. Notebook 05 only identifies the best hyperparameter configuration within each family.

### 4.2 Regularised Linear Model

Ridge Regression is used as the regularised linear candidate. It estimates an additive relationship between the pre-match features and attendance while applying an L2 penalty that shrinks large coefficients.

Regularisation can improve stability when one-hot encoded categories and rolling historical features contain overlapping information. Numerical features use the scaled preprocessing pipeline because the Ridge penalty is sensitive to differences in feature scale.

The regularisation strength is controlled by the `alpha` hyperparameter. Candidate `alpha` values are defined later in Section 4.5.

In [9]:
# Configure the regularised linear candidate.
ridge_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            linear_preprocessor,
        ),
        (
            "model",
            Ridge(),
        ),
    ]
)

# Summarise the Ridge pipeline in a reader-friendly format.
ridge_pipeline_summary = pd.DataFrame(
    [
        {
            "stage": "Numerical preprocessing",
            "inputs": f"{len(NUMERICAL_FEATURES)} numerical features",
            "operation": "Median imputation, then StandardScaler",
            "purpose": "Handle missing values and align feature scales",
        },
        {
            "stage": "Categorical preprocessing",
            "inputs": f"{len(CATEGORICAL_FEATURES)} categorical features",
            "operation": "Most-frequent imputation, then one-hot encoding",
            "purpose": "Convert categories into model-ready columns",
        },
        {
            "stage": "Regression model",
            "inputs": "Transformed feature matrix",
            "operation": "Ridge Regression with L2 regularisation",
            "purpose": "Predict match attendance",
        },
    ]
)

ridge_pipeline_summary

,stage,inputs,operation,purpose
0,Numerical preprocessing,9 numerical features,"Median imputation, then StandardScaler",Handle missing values and align feature scales
1,Categorical preprocessing,8 categorical features,"Most-frequent imputation, then one-hot encoding",Convert categories into model-ready columns
2,Regression model,Transformed feature matrix,Ridge Regression with L2 regularisation,Predict match attendance


### 4.3 Random Forest Regressor

Random Forest Regressor combines predictions from multiple decision trees. Each tree is trained using a different bootstrap sample of the training data and considers a random subset of features when creating splits. Averaging predictions across the trees reduces the instability of an individual decision tree.

Unlike Ridge Regression, Random Forest can capture nonlinear relationships and feature interactions without requiring them to be specified manually. This is useful when the effect of one pre-match feature depends on another, such as a particular home team playing at a particular venue.

The tree-based preprocessing pipeline is used because decision-tree splits do not require numerical feature scaling. Hyperparameters controlling tree quantity, tree depth, leaf size, and feature sampling are defined later in Section 4.5.

In [10]:
# Set a fixed seed for reproducible tree-based model training.
RANDOM_STATE = 42


# Configure the Random Forest candidate.
random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor,
        ),
        (
            "model",
            RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=1,
            ),
        ),
    ]
)


# Summarise the Random Forest pipeline.
random_forest_pipeline_summary = pd.DataFrame(
    [
        {
            "stage": "Numerical preprocessing",
            "inputs": f"{len(NUMERICAL_FEATURES)} numerical features",
            "operation": "Median imputation; no scaling",
            "purpose": "Handle missing numerical values",
        },
        {
            "stage": "Categorical preprocessing",
            "inputs": f"{len(CATEGORICAL_FEATURES)} categorical features",
            "operation": "Most-frequent imputation, then one-hot encoding",
            "purpose": "Convert categories into model-ready columns",
        },
        {
            "stage": "Regression model",
            "inputs": "Transformed feature matrix",
            "operation": "Random Forest Regressor",
            "purpose": "Capture nonlinear relationships and interactions",
        },
    ]
)

random_forest_pipeline_summary

,stage,inputs,operation,purpose
0,Numerical preprocessing,9 numerical features,Median imputation; no scaling,Handle missing numerical values
1,Categorical preprocessing,8 categorical features,"Most-frequent imputation, then one-hot encoding",Convert categories into model-ready columns
2,Regression model,Transformed feature matrix,Random Forest Regressor,Capture nonlinear relationships and interactions


### 4.4 Gradient Boosting Regressor

Gradient Boosting Regressor builds decision trees sequentially. The first tree produces an initial attendance estimate, and each subsequent tree focuses on correcting the prediction errors remaining from the existing ensemble.

This differs from Random Forest, where trees are trained independently and their predictions are averaged. In Gradient Boosting, later trees depend on the errors made by earlier trees, and the contribution of each new tree is controlled by the learning rate.

The model can capture nonlinear relationships and feature interactions without numerical feature scaling. Hyperparameters controlling the number of trees, learning rate, tree depth, and minimum leaf size are defined later in Section 4.5.

In [11]:
# Configure the Gradient Boosting candidate.
gradient_boosting_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            tree_preprocessor,
        ),
        (
            "model",
            GradientBoostingRegressor(
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)


# Summarise the Gradient Boosting pipeline.
gradient_boosting_pipeline_summary = pd.DataFrame(
    [
        {
            "stage": "Numerical preprocessing",
            "inputs": f"{len(NUMERICAL_FEATURES)} numerical features",
            "operation": "Median imputation; no scaling",
            "purpose": "Handle missing numerical values",
        },
        {
            "stage": "Categorical preprocessing",
            "inputs": f"{len(CATEGORICAL_FEATURES)} categorical features",
            "operation": "Most-frequent imputation, then one-hot encoding",
            "purpose": "Convert categories into model-ready columns",
        },
        {
            "stage": "Regression model",
            "inputs": "Transformed feature matrix",
            "operation": "Gradient Boosting Regressor",
            "purpose": "Sequentially correct errors and capture nonlinear patterns",
        },
    ]
)

gradient_boosting_pipeline_summary

,stage,inputs,operation,purpose
0,Numerical preprocessing,9 numerical features,Median imputation; no scaling,Handle missing numerical values
1,Categorical preprocessing,8 categorical features,"Most-frequent imputation, then one-hot encoding",Convert categories into model-ready columns
2,Regression model,Transformed feature matrix,Gradient Boosting Regressor,Sequentially correct errors and capture nonlin...


### 4.5 Hyperparameter Search Spaces

A compact hyperparameter search space is defined for each candidate model family. The ranges are broad enough to compare meaningfully different levels of model complexity while remaining practical across the six time-aware cross-validation folds.

- Ridge Regression is tuned across a logarithmic range of `alpha` values.  
- Random Forest is tuned by varying the `number of trees`, `tree depth`, `minimum leaf size`, and `number of features` considered at each split.  
- Gradient Boosting is tuned by varying the `number of sequential trees`, `learning rate`, `tree depth`, and `minimum leaf size`.

The `model__` prefix tells `GridSearchCV` that each hyperparameter belongs to the final regression model inside the pipeline.

In [12]:
# Define the Ridge Regression search space.
RIDGE_PARAMETER_GRID = {
    "model__alpha": [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0,
        1000.0,
    ],
}


# Define the Random Forest search space.
RANDOM_FOREST_PARAMETER_GRID = {
    "model__n_estimators": [
        300,
        500,
    ],
    "model__max_depth": [
        None,
        12,
        24,
    ],
    "model__min_samples_leaf": [
        1,
        3,
        5,
    ],
    "model__max_features": [
        "sqrt",
        0.7,
    ],
}


# Define the Gradient Boosting search space.
# Define an expanded Gradient Boosting search space.
GRADIENT_BOOSTING_PARAMETER_GRID = {
    "model__n_estimators": [
        100,
        200,
        300,
        500,
    ],
    "model__learning_rate": [
        0.03,
        0.10,
    ],
    "model__max_depth": [
        2,
        3,
        4,
    ],
    "model__min_samples_leaf": [
        2,
        5,
        15,
    ],
}


# Combine each pipeline with its corresponding search space.
MODEL_SEARCH_CONFIGS = {
    "ridge": {
        "model_name": "Ridge Regression",
        "pipeline": ridge_pipeline,
        "parameter_grid": RIDGE_PARAMETER_GRID,
    },
    "random_forest": {
        "model_name": "Random Forest Regressor",
        "pipeline": random_forest_pipeline,
        "parameter_grid": RANDOM_FOREST_PARAMETER_GRID,
    },
    "gradient_boosting": {
        "model_name": "Gradient Boosting Regressor",
        "pipeline": gradient_boosting_pipeline,
        "parameter_grid": GRADIENT_BOOSTING_PARAMETER_GRID,
    },
}


# Summarise the size of each hyperparameter search space.
search_space_records = []

for model_key, config in MODEL_SEARCH_CONFIGS.items():
    parameter_grid = config["parameter_grid"]

    configuration_count = int(
        np.prod(
            [
                len(candidate_values)
                for candidate_values in parameter_grid.values()
            ]
        )
    )

    search_space_records.append(
        {
            "model_key": model_key,
            "model_name": config["model_name"],
            "tuned_hyperparameters": ", ".join(
                parameter_name.removeprefix("model__")
                for parameter_name in parameter_grid
            ),
            "configurations": configuration_count,
            "cv_fits": (
                configuration_count
                * len(time_aware_cv_splits)
            ),
        }
    )


search_space_summary = pd.DataFrame(
    search_space_records
)

search_space_summary

,model_key,model_name,tuned_hyperparameters,configurations,cv_fits
0,ridge,Ridge Regression,alpha,6,36
1,random_forest,Random Forest Regressor,"n_estimators, max_depth, min_samples_leaf, max...",36,216
2,gradient_boosting,Gradient Boosting Regressor,"n_estimators, learning_rate, max_depth, min_sa...",72,432


## 5. Train and Tune Candidate Models

### 5.1 Run the Tuning Process

A separate `GridSearchCV` is run for each candidate model family using the same six time-aware cross-validation folds and the scoring framework defined in Section 3.3.

Within each model family, every hyperparameter configuration is evaluated using both MAE and RMSE. The configuration with the lowest mean CV MAE is retained. Because `refit="mae"` is used, the selected configuration is then refitted automatically on all 1,865 matches in the project-level training partition.

This process tunes each model family independently. It does not compare the three families or select the final forecasting model. The 2024 validation and 2025 locked test partitions remain untouched.

In [13]:
# Run and retain one fitted grid search for each model family.
fitted_searches = {}
tuning_run_records = []


for model_key, config in MODEL_SEARCH_CONFIGS.items():
    model_name = config["model_name"]

    print(f"Starting: {model_name}")

    grid_search = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["parameter_grid"],
        scoring=TUNING_SCORING,
        refit=REFIT_METRIC,
        cv=time_aware_cv_splits,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise",
        verbose=1,
    )

    grid_search.fit(
        X_train,
        y_train,
    )

    fitted_searches[model_key] = grid_search

    configuration_count = len(
        grid_search.cv_results_["params"]
    )

    tuning_run_records.append(
        {
            "model_key": model_key,
            "model_name": model_name,
            "configurations_tested": configuration_count,
            "cv_folds": grid_search.n_splits_,
            "cv_fits": (
                configuration_count
                * grid_search.n_splits_
            ),
            "training_rows": len(X_train),
            "status": "Completed",
        }
    )

    print(f"Completed: {model_name}")
    print()


# Confirm that all three searches completed.
if set(fitted_searches) != set(MODEL_SEARCH_CONFIGS):
    raise RuntimeError(
        "Not all candidate model searches completed successfully."
    )


# Summarise the completed tuning runs.
tuning_run_summary = pd.DataFrame(
    tuning_run_records
)

tuning_run_summary

Starting: Ridge Regression
Fitting 6 folds for each of 6 candidates, totalling 36 fits
Completed: Ridge Regression

Starting: Random Forest Regressor
Fitting 6 folds for each of 36 candidates, totalling 216 fits
Completed: Random Forest Regressor

Starting: Gradient Boosting Regressor
Fitting 6 folds for each of 72 candidates, totalling 432 fits
Completed: Gradient Boosting Regressor



,model_key,model_name,configurations_tested,cv_folds,cv_fits,training_rows,status
0,ridge,Ridge Regression,6,6,36,1865,Completed
1,random_forest,Random Forest Regressor,36,6,216,1865,Completed
2,gradient_boosting,Gradient Boosting Regressor,72,6,432,1865,Completed


### 5.2 Extract the Best Configuration from Each Model Family

The best-MAE configuration is extracted from each fitted grid search. Scikit-learn stores error-based scores as negative values, so the reported MAE and RMSE values are converted back to positive attendee counts.

The RMSE shown for each model corresponds to that model's best-MAE configuration. The refitted `best_estimator_` from each search is retained for later evaluation on the 2024 validation partition.

This section identifies the best configuration within each model family. It does not select a winner across the three model families.

In [14]:
# Retain each refitted candidate and its selected hyperparameters.
best_candidate_estimators = {}
best_parameters_by_model = {}
best_configuration_records = []


for model_key, config in MODEL_SEARCH_CONFIGS.items():
    grid_search = fitted_searches[model_key]
    best_index = grid_search.best_index_
    cv_results = grid_search.cv_results_


    # Confirm that the extracted configuration ranked first by MAE.
    best_mae_rank = int(
        cv_results["rank_test_mae"][best_index]
    )

    if best_mae_rank != 1:
        raise RuntimeError(
            f"{config['model_name']} did not return a "
            "rank-one MAE configuration."
        )


    # Remove the pipeline prefix for reader-friendly reporting.
    selected_parameters = {
        parameter_name.removeprefix("model__"): parameter_value
        for parameter_name, parameter_value
        in grid_search.best_params_.items()
    }


    # Retain the refitted estimator and selected parameters.
    best_candidate_estimators[model_key] = (
        grid_search.best_estimator_
    )

    best_parameters_by_model[model_key] = (
        selected_parameters
    )


    # Extract positive error values for the best-MAE configuration.
    best_configuration_records.append(
        {
            "model_key": model_key,
            "model_name": config["model_name"],
            "best_mean_cv_mae": -float(
                cv_results["mean_test_mae"][best_index]
            ),
            "best_std_cv_mae": float(
                cv_results["std_test_mae"][best_index]
            ),
            "best_mean_cv_rmse": -float(
                cv_results["mean_test_rmse"][best_index]
            ),
            "best_std_cv_rmse": float(
                cv_results["std_test_rmse"][best_index]
            ),
            "selected_hyperparameters": "; ".join(
                f"{name}={value}"
                for name, value in selected_parameters.items()
            ),
        }
    )


# Create the complete results table.
best_configuration_summary = pd.DataFrame(
    best_configuration_records
)


# Create a compact performance table.
best_metrics_display = (
    best_configuration_summary[
        [
            "model_name",
            "best_mean_cv_mae",
            "best_std_cv_mae",
            "best_mean_cv_rmse",
            "best_std_cv_rmse",
        ]
    ]
    .copy()
    .rename(
        columns={
            "model_name": "model",
            "best_mean_cv_mae": "mean_cv_mae",
            "best_std_cv_mae": "std_cv_mae",
            "best_mean_cv_rmse": "mean_cv_rmse",
            "best_std_cv_rmse": "std_cv_rmse",
        }
    )
)

metric_columns = [
    "mean_cv_mae",
    "std_cv_mae",
    "mean_cv_rmse",
    "std_cv_rmse",
]

best_metrics_display[metric_columns] = (
    best_metrics_display[metric_columns]
    .round(2)
)


# Expand the selected parameters into a separate long-format table.
selected_parameter_records = []

for model_key, selected_parameters in best_parameters_by_model.items():
    model_name = MODEL_SEARCH_CONFIGS[
        model_key
    ]["model_name"]

    for parameter_name, selected_value in selected_parameters.items():
        selected_parameter_records.append(
            {
                "model": model_name,
                "hyperparameter": parameter_name,
                "selected_value": str(selected_value),
            }
        )


best_parameters_display = pd.DataFrame(
    selected_parameter_records
)


# Display two compact, reader-friendly tables.
print("Best internal cross-validation performance")
display(best_metrics_display)

print("Selected hyperparameters")
display(best_parameters_display)

Best internal cross-validation performance


,model,mean_cv_mae,std_cv_mae,mean_cv_rmse,std_cv_rmse
0,Ridge Regression,6325.79,228.59,8646.22,307.57
1,Random Forest Regressor,6254.60,358.32,8756.08,418.08
2,Gradient Boosting Regressor,5557.13,204.23,7684.66,312.54


Selected hyperparameters


,model,hyperparameter,selected_value
0,Ridge Regression,alpha,10.0
1,Random Forest Regressor,max_depth,24
2,Random Forest Regressor,max_features,0.7
3,Random Forest Regressor,min_samples_leaf,1
4,Random Forest Regressor,n_estimators,300
5,Gradient Boosting Regressor,learning_rate,0.1
6,Gradient Boosting Regressor,max_depth,3
7,Gradient Boosting Regressor,min_samples_leaf,2
8,Gradient Boosting Regressor,n_estimators,500


## 6. Summarise and Export Candidate Results

### 6.1 Candidate Tuning Summary

Gradient Boosting Regressor produced the strongest internal cross-validation result, with a mean MAE of 5,557 attendees and a mean RMSE of 7,685 attendees. Its mean CV MAE was approximately 12.2% lower than the Ridge Regression benchmark and 11.2% lower than Random Forest.

The expanded Gradient Boosting search also reduced variation in fold-level MAE, with a standard deviation of 204 attendees. This suggests that its improvement was not driven by only one validation season.

These results describe performance within the project-level training partition only. No candidate is promoted in this notebook. The three refitted candidates are retained for independent evaluation on the 2024 validation partition in Notebook 06.

### 6.2 Output Quality Checks

Before export, four concise checks confirm that all three refitted candidates are available, the reported CV metrics are valid, only the fixed training partition has been used, and every candidate pipeline can generate finite predictions.

These checks do not retrain the models or access the 2024 validation and 2025 locked test partitions.

In [15]:
# Define the expected candidate models.
expected_model_keys = set(
    MODEL_SEARCH_CONFIGS
)


# Confirm that all refitted candidates are available.
candidates_complete = (
    set(best_candidate_estimators)
    == expected_model_keys
)


# Confirm that all reported CV metrics are finite.
metric_columns = [
    "best_mean_cv_mae",
    "best_std_cv_mae",
    "best_mean_cv_rmse",
    "best_std_cv_rmse",
]

metric_values = (
    best_configuration_summary[
        metric_columns
    ]
    .to_numpy(dtype="float64")
)

metrics_valid = np.isfinite(
    metric_values
).all()


# Reconfirm that only the fixed training partition was used.
training_partition_valid = (
    len(training_df) == 1865
    and training_df["dataset_role"].eq("training").all()
    and not training_df["season_year"].isin([2024, 2025]).any()
)


# Confirm that each refitted pipeline can produce predictions.
smoke_predictions = {
    model_key: estimator.predict(
        X_train.head(5)
    )
    for model_key, estimator
    in best_candidate_estimators.items()
}

predictions_valid = all(
    len(predictions) == 5
    and np.isfinite(predictions).all()
    for predictions in smoke_predictions.values()
)


# Present the export-readiness checks.
quality_check_summary = pd.DataFrame(
    [
        {
            "check": "Refitted candidates",
            "result": "PASS" if candidates_complete else "FAIL",
        },
        {
            "check": "CV metrics",
            "result": "PASS" if metrics_valid else "FAIL",
        },
        {
            "check": "Training partition",
            "result": "PASS" if training_partition_valid else "FAIL",
        },
        {
            "check": "Prediction smoke test",
            "result": "PASS" if predictions_valid else "FAIL",
        },
    ]
)


if not all(
    [
        candidates_complete,
        metrics_valid,
        training_partition_valid,
        predictions_valid,
    ]
):
    raise RuntimeError(
        "At least one export-readiness check failed."
    )


quality_check_summary

,check,result
0,Refitted candidates,PASS
1,CV metrics,PASS
2,Training partition,PASS
3,Prediction smoke test,PASS


### 6.3 Export Results for Notebook 06

The three refitted candidate pipelines are exported as a single model bundle for evaluation in Notebook 06. Each pipeline contains both its fitted preprocessing steps and its selected regression model.

A compact CSV containing the internal cross-validation metrics and selected hyperparameters is exported alongside the model bundle. Full `GridSearchCV` objects are not saved because they are larger than necessary and are not required for validation-stage evaluation.

These artifacts contain no 2024 validation or 2025 locked test results.

In [16]:
# Stop before export if an output-quality check failed.
if not quality_check_summary["result"].eq("PASS").all():
    raise RuntimeError(
        "Candidate artifacts cannot be exported because "
        "an output-quality check failed."
    )


# Define the candidate-model output locations.
CANDIDATE_MODEL_DIR = (
    project_root
    / "models"
    / "candidates"
)

MODEL_REPORT_DIR = (
    project_root
    / "reports"
    / "modeling"
)

CANDIDATE_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Package the fitted pipelines and their modelling contract.
candidate_model_bundle = {
    "estimators": best_candidate_estimators,
    "selected_parameters": best_parameters_by_model,
    "feature_columns": FEATURE_COLUMNS,
    "numerical_features": NUMERICAL_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "target_column": TARGET_COLUMN,
    "training_seasons": sorted(
        training_df["season_year"]
        .unique()
        .tolist()
    ),
    "cv_validation_seasons": CV_VALIDATION_SEASONS,
    "selection_metric": "MAE",
}


# Define the two exported artifact paths.
candidate_model_path = (
    CANDIDATE_MODEL_DIR
    / "notebook_05_candidate_models.joblib"
)

tuning_summary_path = (
    MODEL_REPORT_DIR
    / "notebook_05_candidate_tuning_summary.csv"
)


# Export the refitted candidate pipelines.
joblib.dump(
    candidate_model_bundle,
    candidate_model_path,
    compress=3,
)


# Export the candidate-level tuning results.
best_configuration_summary.to_csv(
    tuning_summary_path,
    index=False,
)


# Confirm that both artifacts were written successfully.
exported_artifact_summary = pd.DataFrame(
    [
        {
            "artifact": "Candidate model bundle",
            "path": str(
                candidate_model_path.relative_to(
                    project_root
                )
            ),
            "size_kb": round(
                candidate_model_path.stat().st_size
                / 1024,
                1,
            ),
            "status": (
                "Created"
                if candidate_model_path.exists()
                else "Missing"
            ),
        },
        {
            "artifact": "Candidate tuning summary",
            "path": str(
                tuning_summary_path.relative_to(
                    project_root
                )
            ),
            "size_kb": round(
                tuning_summary_path.stat().st_size
                / 1024,
                1,
            ),
            "status": (
                "Created"
                if tuning_summary_path.exists()
                else "Missing"
            ),
        },
    ]
)

exported_artifact_summary

,artifact,path,size_kb,status
0,Candidate model bundle,models\candidates\notebook_05_candidate_models...,10435.4,Created
1,Candidate tuning summary,reports\modeling\notebook_05_candidate_tuning_...,0.6,Created
